### 1. Install dependencies

In [ ]:
%pip install langchain langchain-community langchain-huggingface langchain-chroma langchain-text-splitters sentence-transformers chromadb pypdf -q
dbutils.library.restartPython()


### 2. Load PDFs, chunk, embed, and build the Chroma vector store

In [ ]:
import glob
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

PDF_FOLDER_PATH = "/Volumes/workspace/default/company_policy"
VECTOR_DB_PATH = "/tmp/chroma_policy_db"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# 1. Load all PDFs
pdf_files = glob.glob(f"{PDF_FOLDER_PATH}/*.pdf")
print(f"Found {len(pdf_files)} PDF files.")

pdf_documents = []
for file_path in pdf_files:
    print(f"Loading {file_path}...")
    pdf_documents.extend(PyPDFLoader(file_path).load())
print(f"Loaded {len(pdf_documents)} total pages.")

# 2. Chunk
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=60)
doc_chunks = text_splitter.split_documents(pdf_documents)
print(f"Created {len(doc_chunks)} chunks.")

# 3. Embed + persist to Chroma
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL, model_kwargs={"device": "cpu"})

vector_store = Chroma.from_documents(
    documents=doc_chunks,
    embedding=embeddings,
    persist_directory=VECTOR_DB_PATH
)
print("Vector DB built and saved.")


### 3. Build the LLM + RAG chain, and test it

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    do_sample=False,               # explicit: avoids the temperature/do_sample warning
    return_full_text=False,
    clean_up_tokenization_spaces=False
)
pipe.model.config.max_length = None
llm = HuggingFacePipeline(pipeline=pipe)

SYSTEM_PROMPT = (
    "You are a helpful company policy assistant. Answer the user's question "
    "using ONLY the provided context. If the context does not contain the "
    "answer, state that you do not know."
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def build_prompt(inputs):
    # Uses the model's own chat template instead of hand-written ChatML tags
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{inputs['context']}\n\nQuestion: {inputs['question']}"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RunnableLambda(build_prompt)
    | llm
    | StrOutputParser()
)

print("RAG chain ready!")

# Quick test
question = "Who does the Travel & Expense Reimbursement Policy apply to?"
print(f"Question: {question}\n")
print("Answer:", rag_chain.invoke(question))


### 4. Generate the self-contained `agent.py` used for MLflow serving

In [ ]:
agent_code = """
import pandas as pd
import mlflow
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SYSTEM_PROMPT = (
    "You are a helpful company policy assistant. Answer the user's question "
    "using ONLY the provided context. If the context does not contain the "
    "answer, state that you do not know."
)

def format_docs(docs):
    return "\\n\\n".join(doc.page_content for doc in docs)

class RAGPolicyAgent(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        # vector_db path comes from the bundled MLflow artifact, NOT a hardcoded /tmp path
        embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL, model_kwargs={"device": "cpu"})
        vector_store = Chroma(
            persist_directory=context.artifacts["vector_db"],
            embedding_function=embeddings
        )
        retriever = vector_store.as_retriever(search_kwargs={"k": 3})

        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
        pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=150,
            do_sample=False,
            return_full_text=False,
            clean_up_tokenization_spaces=False
        )
        pipe.model.config.max_length = None
        llm = HuggingFacePipeline(pipeline=pipe)

        def build_prompt(inputs):
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Context:\\n{inputs['context']}\\n\\nQuestion: {inputs['question']}"}
            ]
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        self.rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | RunnableLambda(build_prompt)
            | llm
            | StrOutputParser()
        )

    def predict(self, context, model_input):
        if isinstance(model_input, pd.DataFrame):
            query = model_input["user_message"].iloc[0]
        elif isinstance(model_input, dict):
            query = model_input.get("user_message", "")
        else:
            query = str(model_input)
        return {"response": self.rag_chain.invoke(query)}

mlflow.models.set_model(RAGPolicyAgent())
"""

with open("agent.py", "w") as f:
    f.write(agent_code)

print("agent.py generated successfully.")


### 5. Register the model to Unity Catalog (with the vector DB bundled as an artifact)

In [ ]:
import mlflow
import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

catalog = "workspace"
schema = "default"
model_name = "policy_rag_agent"
full_model_path = f"{catalog}.{schema}.{model_name}"

input_schema = Schema([ColSpec("string", "user_message")])
output_schema = Schema([ColSpec("string", "response")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)
input_example = pd.DataFrame([{"user_message": "What is the travel policy?"}])

with mlflow.start_run(run_name="serving_model_registration"):
    model_info = mlflow.pyfunc.log_model(
        python_model="agent.py",
        artifact_path="agent",
        artifacts={"vector_db": VECTOR_DB_PATH},   # bundles the DB so it's not just a local /tmp path
        signature=signature,
        input_example=input_example,
        registered_model_name=full_model_path
    )
    print(f"Model registered to Unity Catalog: {full_model_path}")
